# Multi-Hop Retrieval — Kaggle Training

**Before running:**
1. Settings → Accelerator → **GPU P100** (or T4)
2. Share your Google Drive `musique_data` folder as **"Anyone with the link can view"**
   - Right-click folder in Drive → Share → change to "Anyone with the link"
   - Copy the folder ID from the URL: `drive.google.com/drive/folders/FOLDER_ID_HERE`
3. Fill in `DRIVE_FOLDER_ID` and `REPO_URL` in the config cell below

**What this does:**
- Clones your GitHub repo
- Downloads MuSiQue data + Model 1 checkpoint from Google Drive via `gdown`
- Trains Model 2 (ColBERTScorer) with Model 1 frozen
- Saves checkpoint to `/kaggle/working/` (auto-saved as Kaggle output)
- Uploads final checkpoint back to Google Drive

In [ ]:
# ── [EDIT THIS] ────────────────────────────────────────────────────────────────
REPO_URL       = "https://github.com/haaaaaaayrewugrhyw/multihop-retrieval.git"
REPO_NAME      = "multihop-retrieval"

# Google Drive folder ID — from URL: drive.google.com/drive/folders/<THIS_PART>
# Folder must be shared as "Anyone with the link can view"
DRIVE_FOLDER_ID = "1mMCf_lhYcw3CH_ttOWDLgSKZuPYqh5m5"   # your musique_data folder

WORK_DIR = f"/kaggle/working/{REPO_NAME}/retrieval"
# ───────────────────────────────────────────────────────────────────────────────

In [ ]:
# 1. Verify GPU
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU. Settings → Accelerator → GPU P100")

print(f"GPU  : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")
print("CUDA OK")

In [ ]:
# 2. Clone repo + install dependencies
import os

repo_root = f"/kaggle/working/{REPO_NAME}"
if not os.path.isdir(repo_root):
    os.system(f"git clone {REPO_URL} {repo_root}")
else:
    os.system(f"cd {repo_root} && git pull")

os.chdir(WORK_DIR)
print("Working dir:", os.getcwd())

# Kaggle already has torch with correct GPU drivers — reinstalling it breaks CUDA.
# Only install the packages that Kaggle's base image doesn't include.
os.system("pip install -q 'transformers>=4.35.0' 'tqdm>=4.65.0' 'numpy>=1.24.0' gdown pydrive2")
print("Dependencies installed")

In [ ]:
# 3. Download files from Google Drive using gdown
import os, gdown

os.makedirs(f"{WORK_DIR}/data/musique", exist_ok=True)
os.makedirs(f"{WORK_DIR}/models", exist_ok=True)
os.makedirs(f"{WORK_DIR}/cache", exist_ok=True)

# Download entire musique_data folder from Drive
# (contains train.jsonl, dev.jsonl, model1_complement.pt)
download_dir = "/kaggle/working/drive_data"
if not os.path.isdir(download_dir):
    print("Downloading from Google Drive...")
    gdown.download_folder(
        id=DRIVE_FOLDER_ID,
        output=download_dir,
        quiet=False,
        use_cookies=False,
    )
else:
    print("Drive data already downloaded")

print("\nDownloaded files:")
for f in sorted(os.listdir(download_dir)):
    size = os.path.getsize(f"{download_dir}/{f}")/1e6
    print(f"  {f:45s}  {size:7.1f} MB")

In [ ]:
# 4. Symlink data files into expected paths
import os

drive = "/kaggle/working/drive_data"

# MuSiQue JSONL → data/musique/
for fname in ["musique_ans_v1.0_train.jsonl", "musique_ans_v1.0_dev.jsonl"]:
    src = f"{drive}/{fname}"
    dst = f"{WORK_DIR}/data/musique/{fname}"
    if not os.path.exists(dst):
        os.symlink(src, dst)
    assert os.path.exists(dst), f"MISSING: {dst}"
    print(f"  {fname}: OK ({os.path.getsize(src)/1e6:.0f} MB)")

# Model 1 checkpoint → models/
m1_src = f"{drive}/model1_complement.pt"
m1_dst = f"{WORK_DIR}/models/model1_complement.pt"
if not os.path.exists(m1_dst):
    os.symlink(m1_src, m1_dst)
assert os.path.exists(m1_dst), f"MISSING: {m1_dst}"
print(f"  model1_complement.pt: OK ({os.path.getsize(m1_src)/1e6:.0f} MB)")

print("\nAll paths ready")

In [ ]:
# 5. Smoke test data loader
import os
os.chdir(WORK_DIR)
os.system("python data_loader.py")

---
## Train Model 2 — ColBERTScorer

- Query encoder from `colbert-ir/colbertv2.0`
- MaxSim scoring: each Q token finds best match in B's complement tokens
- **Model 1 is frozen** — preserves query-agnostic property
- Expected time: **~2.5 hr** on P100 (all 3 epochs)

In [ ]:
# 6. Train Model 2 (full 3-epoch training)
import os
os.chdir(WORK_DIR)
os.system("python model2_train.py")
print("\nModel 2 training complete")

In [ ]:
# 7. Verify checkpoint was saved
import os

best  = f"{WORK_DIR}/models/model2_scorer.pt"
final = f"{WORK_DIR}/models/model2_scorer_final.pt"

for path in [best, final]:
    if os.path.exists(path):
        print(f"  {os.path.basename(path)}: {os.path.getsize(path)/1e6:.1f} MB  OK")
    else:
        print(f"  {os.path.basename(path)}: NOT FOUND")

---
## Save checkpoints back to Google Drive

Uses PyDrive2 with OAuth. When you run the cell below it will print an authorization URL.
Open it in your browser, sign in with the **same Google account as your Drive**, copy the code, and paste it back.

In [ ]:
# 8. Upload checkpoints to Google Drive via PyDrive2
from pydrive2.auth import GoogleAuth
from pydrive2.drive import GoogleDrive
import os

# Auth — opens a browser link, paste the code back here
gauth = GoogleAuth()
gauth.CommandLineAuth()   # prints URL → open in browser → copy code → paste
drive_client = GoogleDrive(gauth)

def upload_to_drive(local_path, drive_folder_id, filename=None):
    fname = filename or os.path.basename(local_path)
    # Remove old version if it exists
    q = f"title='{fname}' and '{drive_folder_id}' in parents and trashed=false"
    existing = drive_client.ListFile({'q': q}).GetList()
    for f in existing:
        f.Delete()
    # Upload new
    f = drive_client.CreateFile({'title': fname, 'parents': [{'id': drive_folder_id}]})
    f.SetContentFile(local_path)
    f.Upload()
    print(f"  Uploaded {fname} ({os.path.getsize(local_path)/1e6:.1f} MB) → Drive")

best  = f"{WORK_DIR}/models/model2_scorer.pt"
final = f"{WORK_DIR}/models/model2_scorer_final.pt"

for ckpt in [best, final]:
    if os.path.exists(ckpt):
        upload_to_drive(ckpt, DRIVE_FOLDER_ID)

print("\nAll checkpoints uploaded to Drive")

In [ ]:
# 9. Final listing of Drive folder
files = drive_client.ListFile(
    {'q': f"'{DRIVE_FOLDER_ID}' in parents and trashed=false"}
).GetList()
print("Drive folder contents:")
for f in sorted(files, key=lambda x: x['title']):
    size_mb = int(f.get('fileSize', 0)) / 1e6
    print(f"  {f['title']:45s}  {size_mb:7.1f} MB")

---
## Done

Checkpoints saved to `musique_data` in your Google Drive:
- `model1_complement.pt` — ComplementEncoder (from previous Colab run)
- `model2_scorer.pt` — ColBERTScorer **best checkpoint**
- `model2_scorer_final.pt` — ColBERTScorer final epoch

To run inference locally:
1. Download both `.pt` files from Drive → `retrieval/models/`
2. Run `python run_full_system.py`